# Indian English + Hindi TTS corpus: a full exploratory tour

This notebook digs into the [Indian English + Hindi TTS Speech Dataset](https://www.kaggle.com/datasets/kavinthakurkm/indian-english-hindi-tts-dataset):
555 single-speaker clips (73.2 minutes) with human-corrected transcripts, per-clip
acoustic metrics, and speaking-style tags.

What we'll cover:

1. Corpus overview and per-language balance
2. Clip duration and where the 4-18 s design window comes from
3. Acoustic quality: SNR and loudness (LUFS)
4. Speaking rate, and why comparing it across scripts is tricky
5. Speaking-style tag distribution
6. ASR vs. human transcript: computing WER from scratch
7. Waveforms and mel spectrograms
8. Listening to the actual audio

The full curation pipeline (VAD segmentation, acoustic QC gates, ASR, human review,
normalization) is open source: https://github.com/auraCodesKM/sarvam-tts-dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd

plt.rcParams["figure.dpi"] = 110

# locate the manifest regardless of how Kaggle mounted the dataset
from pathlib import Path
manifest = next(Path("/kaggle/input").rglob("manifest.csv"))
DATA_DIR = manifest.parent
print("data dir:", DATA_DIR)

df = pd.read_csv(manifest)

# audio_path in the manifest is relative to the original repo; rebuild it from clip_id
df["wav"] = df["clip_id"].map(lambda c: str(DATA_DIR / "clips" / f"{c}.wav"))

print(df.shape)
df.head(3)

## 1. Corpus overview

Two languages, one speaker each. English is an NPTEL engineering lecture
(spontaneous, with natural disfluencies kept in the transcript); Hindi is a studio
reading of a Premchand essay (scripted, much cleaner delivery). That contrast shows
up in almost every plot below.

In [ ]:
overview = df.groupby("language").agg(
    clips=("clip_id", "count"),
    minutes=("duration_s", lambda s: round(s.sum() / 60, 1)),
    mean_dur_s=("duration_s", "mean"),
    median_snr_db=("snr_db", "median"),
    mean_lufs=("lufs", "mean"),
).round(2)
overview

## 2. Clip durations

The pipeline cut clips to sentence-ish units between roughly 4 and 18 seconds -
short enough for TTS training batches, long enough to carry prosody. Worth checking
the two sources ended up with similar shapes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

for lang, sub in df.groupby("language"):
    axes[0].hist(sub["duration_s"], bins=28, alpha=0.6, label=lang)
axes[0].set_xlabel("clip duration (s)")
axes[0].set_ylabel("clips")
axes[0].set_title("Duration distribution")
axes[0].legend()

df.boxplot(column="duration_s", by="language", ax=axes[1])
axes[1].set_title("Duration by language")
axes[1].set_ylabel("seconds")
plt.suptitle("")
plt.tight_layout()
plt.show()

print(df.groupby("language")["duration_s"].describe().round(2))

## 3. Acoustic quality

Every clip carries a signal-to-noise estimate (`snr_db`) and integrated loudness
(`lufs`). The curation pipeline gated on these before any human ever listened, so
the floor should be high - but the *spread* still tells you which source recording
was cleaner.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

for lang, sub in df.groupby("language"):
    axes[0].scatter(sub["snr_db"], sub["lufs"], s=8, alpha=0.5, label=lang)
axes[0].set_xlabel("SNR (dB)")
axes[0].set_ylabel("loudness (LUFS)")
axes[0].set_title("SNR vs loudness")
axes[0].legend()

for lang, sub in df.groupby("language"):
    axes[1].hist(sub["snr_db"], bins=28, alpha=0.6, label=lang)
axes[1].set_xlabel("SNR (dB)")
axes[1].set_title("SNR distribution")
axes[1].legend()
plt.tight_layout()
plt.show()

The Hindi studio narration sits noticeably higher on SNR than the English
lecture-hall recording, which is exactly what you'd expect: one was recorded to be
listened to, the other was recorded to fill a classroom.

## 4. Speaking rate

A crude but useful proxy: transcript characters per second. One caveat before
reading the plot - Devanagari packs more phonetic content per character than Latin
script (consonant clusters, inherent vowels, matras), so a lower chars/sec for
Hindi does **not** mean slower speech. Within-language spread is the meaningful
signal here.

In [ ]:
df["chars_per_s"] = df["text"].str.len() / df["duration_s"]

fig, ax = plt.subplots(figsize=(7, 3.5))
for lang, sub in df.groupby("language"):
    ax.hist(sub["chars_per_s"], bins=30, alpha=0.6, label=lang)
ax.set_xlabel("transcript characters / second")
ax.set_title("Speaking rate proxy (mind the script caveat)")
ax.legend()
plt.show()

print(df.groupby("language")["chars_per_s"].describe().round(2))

## 5. Speaking-style tags

Each clip has a style/emotion tag proposed by an LLM from the transcript text
(the dataset card is upfront that these were not re-verified by ear). Lecture and
essay narration skew heavily neutral - affective tags are rare because the content
is, not because they were dropped.

In [ ]:
ct = pd.crosstab(df["emotion"], df["language"])
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]

ct.plot.bar(figsize=(8, 3.5), width=0.8)
plt.ylabel("clips")
plt.title("Speaking-style tag by language")
plt.tight_layout()
plt.show()
ct

## 6. ASR vs human transcript: WER from scratch

The manifest ships both the raw ASR output (`asr_transcript`) and the
human-corrected `text`, so we can measure how much the reviewer actually had to
fix. Standard word error rate via edit distance - no external deps needed.

In [ ]:
def wer(ref: str, hyp: str) -> float:
    """Word error rate = word-level Levenshtein distance / reference length."""
    r, h = ref.split(), hyp.split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0] = np.arange(len(r) + 1)
    d[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            cost = 0 if r[i - 1] == h[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + cost)
    return d[-1, -1] / max(len(r), 1)

df["wer"] = [wer(t, a) for t, a in zip(df["text"], df["asr_transcript"])]

summary = df.groupby("language").agg(
    clips=("clip_id", "count"),
    edited_pct=("edited", lambda s: round(100 * s.mean(), 1)),
    mean_wer=("wer", lambda s: round(s.mean(), 4)),
    max_wer=("wer", lambda s: round(s.max(), 4)),
)
summary

In [ ]:
# what did the reviewer actually change? show the edited clips
edited = df[df["edited"] == True]
print(f"{len(edited)} of {len(df)} clips were edited by the human reviewer\n")
for _, row in edited.iterrows():
    print(f"--- {row.clip_id} ({row.language})")
    print(f"ASR:   {row.asr_transcript}")
    print(f"final: {row.text}\n")

Near-zero WER against the *corrected* reference is expected by construction -
the human started from the ASR output and only fixed real mistakes. The interesting
number is the edit rate: how often ASR was already good enough to ship untouched.

## 7. Waveforms and mel spectrograms

Numbers are fine, but for a TTS dataset you want to see the audio. One clip per
language: waveform on top, log-mel spectrogram below.

In [ ]:
def show_clip(row):
    y, sr = librosa.load(row.wav, sr=None)
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)

    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"{row.clip_id} ({row.language}, {row.duration_s:.1f}s, SNR {row.snr_db} dB)")

    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80, fmax=sr // 2)
    librosa.display.specshow(librosa.power_to_db(S, ref=np.max), sr=sr,
                             x_axis="time", y_axis="mel", fmax=sr // 2, ax=axes[1])
    axes[1].set_title("log-mel spectrogram (80 mels)")
    plt.tight_layout()
    plt.show()

for lang in ["en-IN", "hi-IN"]:
    # pick a mid-length, high-SNR clip rather than just .iloc[0]
    sub = df[df.language == lang]
    pick = sub[(sub.duration_s.between(8, 12))].nlargest(1, "snr_db").iloc[0]
    show_clip(pick)

Clean harmonic structure, no visible broadband noise floor, tight
silence-trimming at the edges - the QC gates did their job.

## 8. Listen for yourself

In [ ]:
for lang in ["en-IN", "hi-IN"]:
    sub = df[df.language == lang]
    row = sub[sub.duration_s.between(8, 14)].nlargest(1, "snr_db").iloc[0]
    print(f"{lang}: {row.text}")
    display(ipd.Audio(row.wav))
    print()

## Wrap-up

- 555 clips / 73.2 min, near 50/50 English-Hindi split, one speaker per language
- High, tightly-distributed SNR; loudness-normalized around -19 LUFS
- Both raw ASR and corrected transcripts included, so the set doubles as a small
  ASR benchmark
- Style tags are transcript-derived and skew neutral (lecture/narration content)

Some things to try from here:

- Fine-tune a small TTS model (VITS, StyleTTS2) on one of the speakers
- Use the (asr_transcript, text) pairs to study ASR error patterns on Indian English
- Benchmark a multilingual ASR model (Whisper, IndicConformer) against the
  human-corrected references

Pipeline code and the full quality report: https://github.com/auraCodesKM/sarvam-tts-dataset

If this was useful, an upvote on the dataset helps other people working on
Indian-language speech find it.